# RODNet 학습 (공식 `train.py`)

`run_rodnet_train.py`와 동일한 파이프라인을 노트북에서 실행합니다.

1. TRAIN 라벨 + RAMap → `rodnet_staging_train_rod2021/`
2. `prepare_data.py --split train` → `.pkl`
3. `tools/train.py` → `data/rodnet_checkpoints_train/` 아래 로그·체크포인트

**사전:** `cruw-devkit`·`RODNet` editable 설치 (`1_rodnet_demo.ipynb` 절차와 동일).

## 진행 중인지 확인하는 방법

| 방법 | 설명 |
|------|------|
| **아래 §1 셀** | `train.log` 마지막 줄·파일 수정 시각 |
| **작업 관리자** (Windows) | `python.exe` CPU·메모리 사용 여부 |
| **GPU** | 터미널에서 `nvidia-smi` — Python이 VRAM을 쓰는지 |
| **폴더** | `data/rodnet_checkpoints_train/` 에 실험 폴더·`epoch_*_final.pkl` |

터미널에서 백그라운드로 돌려도 **같은 로그 파일**을 §1에서 볼 수 있습니다.

In [4]:
from __future__ import annotations

import os
import sys
from pathlib import Path

RADAR_CRUW = Path.cwd().resolve()
if not (RADAR_CRUW / "run_rodnet_train.py").is_file():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ai" / "radar-cruw" / "run_rodnet_train.py").is_file():
            RADAR_CRUW = p / "ai" / "radar-cruw"
            break
if str(RADAR_CRUW) not in sys.path:
    sys.path.insert(0, str(RADAR_CRUW))

DATA_DIR = Path(os.environ.get("CRUW_DATA_DIR", "").strip() or (RADAR_CRUW / "data"))

from run_rodnet_train import print_training_status, run_training

print("RADAR_CRUW:", RADAR_CRUW)
print("DATA_DIR:", DATA_DIR)

RADAR_CRUW: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw
DATA_DIR: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data


## 1. 상태 확인 (`train.log` 마지막 줄)

학습이 돌아가면 이 로그에 `epoch ... loss` 줄이 추가됩니다. **학습 중에도 셀을 다시 실행**해 갱신할 수 있습니다.

In [5]:
print_training_status(DATA_DIR, tail_lines=40)

torch.cuda.is_available(): False
checkpoint 루트: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_checkpoints_train | exists: True
가장 최근 train.log: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_checkpoints_train\rodnet-cdc-win16-20260325-165342\train.log
수정 시각: 2026-03-25 16:53:50.079263
--- 마지막 40 줄 ---
epoch  1, iter    1: loss: 0.6986 (0.6986) | load time: 0.26 | back time: 4.31


## 2. 학습 실행

`EPOCHS`, `STEM` 등을 바꾼 뒤 셀 실행. **한 셀이 끝날 때까지** 대기합니다 (CPU면 매우 느릴 수 있음).

이미 스테이징·pkl이 있으면 `SKIP_STAGE = True`, `SKIP_PREPARE = True` 로 학습만 반복할 수 있습니다.

In [6]:
# 파라미터
EPOCHS = 1
BATCH_SIZE = 2
LOG_STEP = 50
STEM = ""  # 비우면 자동: 짝 맞는 첫 TRAIN 시퀀스
SKIP_STAGE = False
SKIP_PREPARE = False

result = run_training(
    data_dir=DATA_DIR,
    stem=STEM,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    log_step=LOG_STEP,
    skip_stage=SKIP_STAGE,
    skip_prepare=SKIP_PREPARE,
)
result

스테이징: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_staging_train_rod2021 시퀀스: 2019_04_09_BMS1000
실행: c:\Users\taehu\AppData\Local\Programs\Python\Python310\python.exe C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet\tools\prepare_dataset\prepare_data.py --config C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet\configs\config_rodnet_cdc_win16.py --data_root C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_staging_train_rod2021 --split train --out_data_dir C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet\data\prepared_train_project --overwrite
Preparing train sets ...
Sequence C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_staging_train_rod2021/sequences\train\2019_04_09_BMS1000 saving to C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet\data\prepared_train_project\train\2019_04_09_BMS1000.pkl
생성된 pkl: 1
실행: c:\Users\taehu\

{'staging': WindowsPath('C:/Users/taehu/Desktop/projects/hanhwa_final/ai/radar-cruw/data/rodnet_staging_train_rod2021'),
 'prepared': WindowsPath('C:/Users/taehu/Desktop/projects/hanhwa_final/ai/radar-cruw/vendor/RODNet/data/prepared_train_project'),
 'log_dir': WindowsPath('C:/Users/taehu/Desktop/projects/hanhwa_final/ai/radar-cruw/data/rodnet_checkpoints_train'),
 'stem': '2019_04_09_BMS1000',
 'config': WindowsPath('C:/Users/taehu/Desktop/projects/hanhwa_final/ai/radar-cruw/vendor/RODNet/configs/config_rodnet_cdc_win16.py')}